# Exploración ICS e IPM

Carga de shapefiles desde el repositorio, visualización en mapa y análisis inicial del Índice de Condición Social (ICS) y el Índice de Pobreza Multidimensional (IPM) a nivel de manzana.

In [ ]:
# @title 1. Montar Google Drive (opcional, solo en Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive montado')
else:
    print('Ejecutando localmente')

In [ ]:
# @title 2. Instalar dependencias
!pip install geopandas matplotlib mapclassify pyarrow -q

In [ ]:
# @title 3. Clonar el repositorio (solo si es necesario)
import os
import shutil

REPO_URL = 'https://github.com/j0rg3c45/Pobreza_multidimensional_y_condicion_social.git'
REPO_DIR = 'Pobreza_multidimensional_y_condicion_social'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull
    %cd ..
print('Repositorio listo')

In [ ]:
# @title 4. Importar librerías
import zipfile
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

print('Librerías importadas')

In [ ]:
# @title 5. Definir rutas y descomprimir
BASE_DIR = REPO_DIR if os.path.exists(REPO_DIR) else '.'
DATA_DIR = os.path.join(BASE_DIR, 'indice_Pobreza', 'data')

def extract_zip(zip_name):
    zip_path = os.path.join(DATA_DIR, zip_name)
    out_dir = os.path.join(DATA_DIR, zip_name.replace('.zip', ''))
    if not os.path.exists(out_dir):
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(out_dir)
        print(f'Extraído: {zip_name} → {out_dir}')
    else:
        print(f'Ya existe: {out_dir}')
    return out_dir

ics_dir = extract_zip('ICS.zip')
ipm_dir = extract_zip('IPM.zip')

In [ ]:
# @title 6. Cargar shapefiles
gdf_ics = gpd.read_file(os.path.join(ics_dir, 'Mzn_ics.shp'))
gdf_ipm = gpd.read_file(os.path.join(ipm_dir, 'Mzn_ipm.shp'))

print(f'ICS: {gdf_ics.shape[0]} geometrías, {gdf_ics.shape[1]} columnas')
print(f'IPM: {gdf_ipm.shape[0]} geometrías, {gdf_ipm.shape[1]} columnas')

In [ ]:
# @title 7. Ver columnas disponibles
print('=== Columnas ICS ===')
print(gdf_ics.columns.tolist())
print()
print('=== Columnas IPM ===')
print(gdf_ipm.columns.tolist())

In [ ]:
# @title 8. Vista previa de los datos
print('=== ICS (primeras 5 filas) ===')
display(gdf_ics.head())
print()
print('=== IPM (primeras 5 filas) ===')
display(gdf_ipm.head())

In [ ]:
# @title 9. Sistema de referencia de coordenadas (CRS)
print(f'ICS CRS: {gdf_ics.crs}')
print(f'IPM CRS: {gdf_ipm.crs}')

In [ ]:
# @title 10. Mapa base — Manzanas ICS
fig, ax = plt.subplots(1, 1, figsize=(14, 12))
gdf_ics.plot(ax=ax, color='#e0e0e0', edgecolor='#999999', linewidth=0.3, alpha=0.7)
ax.set_title('Manzanas — ICS (Índice de Condición Social)', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# @title 11. Mapa base — Manzanas IPM
fig, ax = plt.subplots(1, 1, figsize=(14, 12))
gdf_ipm.plot(ax=ax, color='#e0e0e0', edgecolor='#999999', linewidth=0.3, alpha=0.7)
ax.set_title('Manzanas — IPM (Índice de Pobreza Multidimensional)', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# @title 12. Identificar columna de ICS para clasificación
# Buscar columna numérica que contenga el valor del ICS
ics_cols = [c for c in gdf_ics.columns if gdf_ics[c].dtype in ['float64', 'int64']]
print('Columnas numéricas en ICS:', ics_cols)

# Mostrar distribución de las primeras columnas numéricas
for col in ics_cols[:5]:
    print(f'\n{col}:')
    print(gdf_ics[col].describe())

In [ ]:
# @title 13. Mapa temático — ICS (si existe columna de valor)
# Identificar columna candidata para mapeo
num_cols_ics = [c for c in gdf_ics.columns if gdf_ics[c].dtype in ['float64', 'int64'] and c.lower() not in ['objectid', 'fid', 'id', 'shape_leng', 'shape_area', 'shape_len', 'shape_le']]

if num_cols_ics:
    col = num_cols_ics[0]
    fig, ax = plt.subplots(1, 1, figsize=(14, 12))
    gdf_ics.plot(column=col, ax=ax, legend=True,
                 cmap='viridis', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': col, 'shrink': 0.6})
    ax.set_title(f'ICS — {col}', fontsize=14, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
else:
    print('No se encontró columna numérica para mapear en ICS')

In [ ]:
# @title 14. Mapa temático — IPM
num_cols_ipm = [c for c in gdf_ipm.columns if gdf_ipm[c].dtype in ['float64', 'int64'] and c.lower() not in ['objectid', 'fid', 'id', 'shape_leng', 'shape_area', 'shape_len', 'shape_le']]

if num_cols_ipm:
    col = num_cols_ipm[0]
    fig, ax = plt.subplots(1, 1, figsize=(14, 12))
    gdf_ipm.plot(column=col, ax=ax, legend=True,
                 cmap='RdYlGn_r', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': col, 'shrink': 0.6})
    ax.set_title(f'IPM — {col}', fontsize=14, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
else:
    print('No se encontró columna numérica para mapear en IPM')

In [ ]:
# @title 15. Mapas lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 14))

# ICS
if num_cols_ics:
    gdf_ics.plot(column=num_cols_ics[0], ax=ax1, legend=True,
                 cmap='viridis', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': num_cols_ics[0], 'shrink': 0.5})
    ax1.set_title('ICS', fontsize=13, fontweight='bold')
else:
    gdf_ics.plot(ax=ax1, color='#e0e0e0', edgecolor='#999999', linewidth=0.2)
    ax1.set_title('ICS (sin datos)', fontsize=13, fontweight='bold')
ax1.set_axis_off()

# IPM
if num_cols_ipm:
    gdf_ipm.plot(column=num_cols_ipm[0], ax=ax2, legend=True,
                 cmap='RdYlGn_r', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': num_cols_ipm[0], 'shrink': 0.5})
    ax2.set_title('IPM', fontsize=13, fontweight='bold')
else:
    gdf_ipm.plot(ax=ax2, color='#e0e0e0', edgecolor='#999999', linewidth=0.2)
    ax2.set_title('IPM (sin datos)', fontsize=13, fontweight='bold')
ax2.set_axis_off()

plt.suptitle('Comparación ICS vs IPM por Manzana', fontsize=16, fontweight='bold', y=0.92)
plt.tight_layout()
plt.show()

In [ ]:
# @title 16. Estadísticas descriptivas
print('=== ICS ===')
if num_cols_ics:
    display(gdf_ics[num_cols_ics].describe())

print('\n=== IPM ===')
if num_cols_ipm:
    display(gdf_ipm[num_cols_ipm].describe())